## 1. Convert csv Files to Parquet Format

In [10]:
import os
import glob
import pandas as pd

# Root folder that contains subfolders of CSV files
root_dir = r"C:/Users/ekadw/Documents/DATA/Stock_Market/"

# Recursively find all CSV files
files = glob.glob(os.path.join(root_dir, "**", "*.csv"), recursive=True)

print(f"Found {len(files)} CSV files")

for f in files:
    try:
        # Read CSV
        df = pd.read_csv(f)
        
        # Build matching parquet path (same folder, same name, .parquet extension)
        parquet_path = os.path.splitext(f)[0] + ".parquet"
        
        # Save to Parquet
        df.to_parquet(parquet_path, index=False)
        
        #print(f"Converted: {f} -> {parquet_path}") # it is too long
    except Exception as e:
        print(f"Failed to convert {f}: {e}")


Found 8050 CSV files


## 2. Use SQL query to produce the fDD dataframe the same as assignment 1

In [8]:
import duckdb
con = duckdb.connect()

stock_query = """
WITH CTE_filename AS (
    SELECT
        *,
        "Adj Close" AS Adj_Close,
        regexp_extract(filename, '([^/\\\\]+)\\.parquet$') AS source,
        regexp_extract(filename, '([^/\\\\]+)\\.parquet$', 1) AS ticker,
        EXTRACT(YEAR FROM CAST(Date AS DATE)) AS year,
        LAG(Close, 1) OVER (PARTITION BY ticker ORDER BY Date ASC) AS Close_Lag_1,
        LAG(Adj_Close,1) OVER (PARTITION BY ticker ORDER BY Date ASC) AS Adj_Close_Lag_1,
        High - Low AS hi_lo_range
    FROM read_parquet('C:/Users/ekadw/Documents/DATA/Stock_Market/*/*.parquet', filename=true)
), CTE_lag AS (
    SELECT
        *,
        (Close/Close_Lag_1) - 1 AS returns
    FROM CTE_filename
), CTE_returns AS (
    SELECT
        *,
        CASE 
            WHEN COUNT(returns) OVER (PARTITION BY ticker ORDER BY Date ROWS BETWEEN 9 PRECEDING AND CURRENT ROW) = 10
            THEN AVG(returns) OVER (PARTITION BY ticker ORDER BY Date ROWS BETWEEN 9 PRECEDING AND CURRENT ROW)
        END AS ma10_returns
    FROM CTE_lag
)

SELECT
    Date,
    Open,
    High,
    Low,
    Close,
    Adj_Close,
    Volume,
    source,
    ticker,
    year,
    Close_Lag_1,
    Adj_Close_Lag_1,
    returns,
    hi_lo_range,
    ma10_returns
FROM CTE_returns
"""
stock_market = con.execute(stock_query).fetchdf()
stock_market.head()

,Date,Open,High,Low,Close,Adj_Close,Volume,source,ticker,year,Close_Lag_1,Adj_Close_Lag_1,returns,hi_lo_range,ma10_returns
0,2000-10-26,10.330579,10.330579,10.330579,10.330579,5.073581,3900,CTBI.parquet,CTBI,2000,10.373267,5.094546,-0.004115,0.000000,-0.000755
1,2000-10-27,10.330579,10.330579,10.330579,10.330579,5.073581,11000,CTBI.parquet,CTBI,2000,10.330579,5.073581,0.000000,0.000000,-0.001574
2,2000-10-30,10.330579,10.415956,10.159825,10.394611,5.105032,16500,CTBI.parquet,CTBI,2000,10.330579,5.073581,0.006198,0.256130,0.001484
3,2000-10-31,10.245202,10.245202,10.074449,10.159825,4.989721,13400,CTBI.parquet,CTBI,2000,10.394611,5.105032,-0.022587,0.170753,0.000476
4,2000-11-01,10.159825,10.159825,10.159825,10.159825,4.989721,0,CTBI.parquet,CTBI,2000,10.159825,4.989721,0.000000,0.000000,0.000476


## 3.Check the part of dataframe that shows the result is the same with assignment 1

In [9]:
fDD = stock_market[stock_market["ticker"]=='AAON'].sort_values(by="Date")
fDD.head(20)

,Date,Open,High,Low,Close,Adj_Close,Volume,source,ticker,year,Close_Lag_1,Adj_Close_Lag_1,returns,hi_lo_range,ma10_returns
13928192,1992-12-16,0.099763,0.099763,0.099763,0.099763,0.084743,1900,AAON.parquet,AAON,1992,NaN,NaN,NaN,0.000000,NaN
13928193,1992-12-17,0.119716,0.129692,0.099763,0.119716,0.101691,163900,AAON.parquet,AAON,1992,0.099763,0.084743,0.200000,0.029929,NaN
13928194,1992-12-18,0.119716,0.119716,0.119716,0.119716,0.101691,14100,AAON.parquet,AAON,1992,0.119716,0.101691,0.000000,0.000000,NaN
13928195,1992-12-21,0.099763,0.099763,0.089787,0.099763,0.084743,32400,AAON.parquet,AAON,1992,0.119716,0.101691,-0.166667,0.009976,NaN
13928196,1992-12-22,0.099763,0.119716,0.099763,0.119716,0.101691,62100,AAON.parquet,AAON,1992,0.099763,0.084743,0.200000,0.019953,NaN
13928197,1992-12-23,0.099763,0.099763,0.099763,0.099763,0.084743,14100,AAON.parquet,AAON,1992,0.119716,0.101691,-0.166667,0.000000,NaN
13928198,1992-12-24,0.129692,0.139668,0.129692,0.129692,0.110165,62100,AAON.parquet,AAON,1992,0.099763,0.084743,0.300000,0.009976,NaN
13928199,1992-12-28,0.129692,0.129692,0.129692,0.129692,0.110165,62100,AAON.parquet,AAON,1992,0.129692,0.110165,0.000000,0.000000,NaN
13928200,1992-12-29,0.129692,0.129692,0.129692,0.129692,0.110165,0,AAON.parquet,AAON,1992,0.129692,0.110165,0.000000,0.000000,NaN
13928201,1992-12-30,0.129692,0.129692,0.129692,0.129692,0.110165,0,AAON.parquet,AAON,1992,0.129692,0.110165,0.000000,0.000000,NaN


In [10]:
len(stock_market)

28151758